# ⚽ Projeto 2: Esportes & Futebol
## Mesa Redonda Analitica - Deteccao de Clima de Crise em Clubes

**Disciplina:** Linguagens de Programacao para Engenharia de Dados  
**Instituicao:** UNIFOR  
**Professor:** Cassio Pinheiro

---

**Cenario de negocio:** Agencias de marketing esportivo monitoram a flutuacao do humor e das criticas a tecnicos e jogadores logo apos as rodadas do Brasileirao, detectando 'clima de crise' antecipadamente.

**Canais sugeridos:** canais de mesa redonda e analise pos-jogo

### Arquitetura do pipeline
Este notebook implementa um lakehouse local em tres camadas, **totalmente executavel** (com fallback sintetico determinista caso nao haja IDs validos ou legendas disponiveis):

| Camada | Responsabilidade |
|---|---|
| **Bronze** | Extracao bruta das transcricoes (resiliente a falhas) |
| **Silver** | Limpeza, contrato Pandera e quarentena de descartes |
| **Gold** | Particionamento por clima (crise vs positivo) com Polars Lazy |


### 1. Setup do ambiente

Rode esta atividade no **VS Code** com a extensao *Jupyter*. Antes de executar, instale as dependencias **uma unica vez** num ambiente virtual:

```bash
python -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```

No VS Code, selecione o kernel desse `.venv` no canto superior direito do notebook e execute as celulas em ordem (Bronze -> Silver -> Gold).

In [ ]:
# === SETUP DO AMBIENTE ===
# Execute uma unica vez por sessao do kernel (dependencias: requirements.txt).

import os, re, json, random, warnings
from datetime import datetime, timedelta, timezone

import pandas as pd
import polars as pl
import duckdb
import pandera.pandas as pa
from pandera.pandas import Column, DataFrameSchema, Check

warnings.filterwarnings("ignore")
random.seed(42)  # determinismo: a aula roda igual em todas as maquinas

# Pastas das camadas (data lakehouse local, criadas na pasta do notebook)
for layer in ("bronze", "silver", "silver/_quarentena", "gold"):
    os.makedirs(f"./datalake/{layer}", exist_ok=True)

print("Ambiente pronto. pandas", pd.__version__, "| polars", pl.__version__,
      "| duckdb", duckdb.__version__)

### 2. Bronze - extracao das transcricoes
Cole IDs reais em `VIDEO_IDS`. Sem IDs (ou sem legenda), o pipeline usa dados sinteticos deterministicos e roda do mesmo jeito.

In [ ]:
# === CAMADA BRONZE: extracao resiliente de transcricoes ===
# Cole abaixo IDs reais de videos do dominio "futebol".
# (o ID e o trecho apos v= na URL: youtube.com/watch?v=ESTE_AQUI)
VIDEO_IDS = []   # ex.: ["dQw4w9WgXcQ", "9bZkp7q19f0"]

VOCABULARIO = ["crise", "demissao", "vexame", "pressao", "titulo", "vitoria", "tecnico"]  # termos-alvo deste dominio
IDIOMAS = ["pt", "pt-BR", "en"]

def extrair_transcricao(video_id):
    """Retorna lista de trechos {text,start,duration} ou None se indisponivel."""
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        api = YouTubeTranscriptApi()                       # API v1.x (instancia)
        fetched = api.fetch(video_id, languages=IDIOMAS)
        return [{"text": s.text, "start": s.start, "duration": s.duration}
                for s in fetched]
    except Exception as e:
        print(f"  [aviso] {video_id}: legenda indisponivel ({type(e).__name__}).")
        return None

def gerar_sintetico(video_id, n_trechos=180):
    """Fallback DETERMINISTICO: garante que a aula roda sem depender de rede."""
    rng = random.Random(hash(video_id) & 0xFFFFFFFF)
    base = ["entao", "olha", "o ponto principal aqui", "veja bem", "isso significa",
            "na pratica", "vale destacar", "o dado mostra que", "repare que"]
    trechos, t = [], 0.0
    for _ in range(n_trechos):
        palavras = rng.sample(base, k=2)
        # injeta termos-alvo do dominio com probabilidade controlada
        if rng.random() < 0.30:
            palavras.append(rng.choice(VOCABULARIO))
        dur = round(rng.uniform(2.0, 6.0), 2)
        trechos.append({"text": " ".join(palavras), "start": round(t, 2),
                        "duration": dur})
        t += dur
    return trechos

def ingerir(video_ids):
    registros, origem_sintetica = [], False
    alvos = video_ids if video_ids else [f"demo_{i:02d}" for i in range(3)]
    for vid in alvos:
        trechos = extrair_transcricao(vid) if video_ids else None
        if trechos is None:
            trechos = gerar_sintetico(vid)
            origem_sintetica = True
        for ordem, tr in enumerate(trechos):
            registros.append({"video_id": vid, "ordem": ordem,
                              "texto": tr["text"], "start": tr["start"],
                              "duration": tr["duration"]})
    df = pd.DataFrame(registros)
    if origem_sintetica:
        print(">> Usando dados SINTETICOS (sem IDs validos ou sem legenda). "
              "Pipeline 100% funcional para a aula.")
    df.to_parquet("./datalake/bronze/transcricoes.parquet", index=False)
    return df

bronze = ingerir(VIDEO_IDS)
print(f"BRONZE: {len(bronze)} trechos de {bronze.video_id.nunique()} video(s).")
bronze.head()

### 3. Silver - limpeza, contrato e quarentena
Aplica limpeza vetorizada, valida o **contrato de dados** com Pandera e isola descartes numa area de quarentena.

In [ ]:
# === CAMADA SILVER: limpeza, contrato (Pandera) e quarentena ===
STOPWORDS = {"entao", "olha", "ne", "tipo", "veja", "bem", "o", "a", "que",
             "de", "e", "aqui", "isso"}

def limpar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-zaaaeeiooouuc\s]", " ", texto)
    tokens = [t for t in texto.split() if t not in STOPWORDS and len(t) > 2]
    return " ".join(tokens)

df = bronze.copy()
df["texto_limpo"] = df["texto"].apply(limpar)
df["n_palavras"]  = df["texto_limpo"].str.split().str.len().fillna(0).astype(int)

# Regra de quarentena: trecho vazio apos limpeza ou curto demais
mask_ok = df["n_palavras"] >= 1
quarentena = df[~mask_ok].copy()
silver = df[mask_ok].copy()

if len(quarentena):
    quarentena.to_parquet("./datalake/silver/_quarentena/descartes.parquet",
                          index=False)

# Contrato de dados: o schema documenta E impoe as garantias da camada Silver
schema = DataFrameSchema({
    "video_id":    Column(str, nullable=False),
    "ordem":       Column(int, Check.ge(0)),
    "texto_limpo": Column(str, Check.str_length(min_value=1)),
    "start":       Column(float, Check.ge(0)),
    "duration":    Column(float, Check.gt(0)),
    "n_palavras":  Column(int, Check.ge(1)),
}, coerce=True)

silver = schema.validate(silver)
silver.to_parquet("./datalake/silver/transcricoes.parquet", index=False)
print(f"SILVER: {len(silver)} validos | QUARENTENA: {len(quarentena)} descartados.")
silver.head()

### 4. Gold - Particionamento por clima (crise vs positivo) com Polars Lazy

In [ ]:
# === CAMADA GOLD: Polars Lazy + particionamento Parquet por clima ===
NEG = {"crise", "demissao", "vexame", "pressao"}
POS = {"titulo", "vitoria"}

lz = pl.scan_parquet("./datalake/silver/transcricoes.parquet")

gold = (
    lz.with_columns([
        pl.col("texto_limpo").str.split(" ").alias("tokens")
    ])
    .with_columns([
        pl.col("tokens").list.eval(
            pl.element().is_in(list(NEG))).list.sum().alias("hits_neg"),
        pl.col("tokens").list.eval(
            pl.element().is_in(list(POS))).list.sum().alias("hits_pos"),
    ])
    .with_columns(
        pl.when(pl.col("hits_neg") > pl.col("hits_pos")).then(pl.lit("crise"))
          .when(pl.col("hits_pos") > pl.col("hits_neg")).then(pl.lit("positivo"))
          .otherwise(pl.lit("neutro")).alias("clima")
    )
    .group_by(["video_id", "clima"])
    .agg(pl.len().alias("trechos"))
    .sort(["video_id", "trechos"], descending=[False, True])
    .collect()
)
gold.write_parquet("./datalake/gold/clima_clubes.parquet")
print("GOLD: distribuicao de clima por video (Polars Lazy)")
print(gold)

---
### Desafios de extensao (substituem os antigos `# TODO`)

Estes blocos sao **opcionais** e servem para aprofundar. O pipeline acima ja
roda completo; aqui voce estende.

1. **Bronze:** trate o caso de playlist (varios videos) com `try/except` por video
   e registre num log de erros quantos falharam.
2. **Silver:** adicione ao contrato Pandera um `Check` de unicidade da chave
   composta `(video_id, ordem)` usando `Check(lambda df: ~df.duplicated())`.
3. **Gold:** parametrize o `VOCABULARIO` por arquivo `.json` externo e recalcule
   a analise sem alterar o codigo SQL.

> Rastreabilidade: este projeto exercita os conceitos de Bronze/Silver/Gold aplicados ao dominio de Esportes & Futebol.